# Lesson 1: Named Entity Recognition (NER) - Fundamentals & First Principles

![NER](https://upload.wikimedia.org/wikipedia/commons/thumb/3/3f/NER_example.svg/1200px-NER_example.svg.png)

## 🎯 Learning Objectives

By the end of this lesson, you will:
1. Understand what Named Entity Recognition is and why it matters
2. Learn the core concepts: entities, tags, and tagging schemes
3. Understand the BIO/IOB tagging scheme in depth
4. Explore different NER approaches and their trade-offs
5. Build intuition for how NER models work

---

## 📚 Table of Contents

1. [What is Named Entity Recognition?](#1-what-is-named-entity-recognition)
2. [Why NER Matters](#2-why-ner-matters)
3. [Entity Types & Categories](#3-entity-types--categories)
4. [Tagging Schemes: BIO, BIOES, and IOB](#4-tagging-schemes-bio-bioes-and-iob)
5. [Evolution of NER Approaches](#5-evolution-of-ner-approaches)
6. [Hands-on: Basic NER Exploration](#6-hands-on-basic-ner-exploration)
7. [Further Reading & Resources](#7-further-reading--resources)

---

## 📦 Setup & Installation

Run the following cell to install all required packages for this lesson:

In [ ]:
# Install required packages
!pip install -q spacy datasets transformers seqeval
!python -m spacy download en_core_web_sm -q

In [ ]:
# Import libraries
import spacy
from collections import Counter
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')

print("✅ All imports successful!")

---

## 1. What is Named Entity Recognition?

### Definition

> **Named Entity Recognition (NER)** is a subtask of information extraction that seeks to locate and classify named entities mentioned in unstructured text into pre-defined categories such as person names, organizations, locations, medical codes, time expressions, quantities, monetary values, percentages, etc.
>
> — [Wikipedia: Named-entity recognition](https://en.wikipedia.org/wiki/Named-entity_recognition)

### First Principles Understanding

At its core, NER answers the question: **"What are the important 'things' mentioned in this text, and what type of 'things' are they?"**

Consider this sentence:

```
"Apple Inc. was founded by Steve Jobs in Cupertino, California on April 1, 1976."
```

A human reading this immediately recognizes:
- **Apple Inc.** → A company (ORGANIZATION)
- **Steve Jobs** → A person (PERSON)
- **Cupertino** → A city (LOCATION)
- **California** → A state (LOCATION)
- **April 1, 1976** → A date (DATE)

NER systems attempt to replicate this human ability to identify and categorize named entities.

### Formal Definition

Mathematically, NER can be formulated as:

Given a sequence of tokens $X = (x_1, x_2, ..., x_n)$, predict a corresponding sequence of labels $Y = (y_1, y_2, ..., y_n)$ where each $y_i$ belongs to a predefined set of entity tags.

This makes NER a **sequence labeling** problem, similar to:
- Part-of-Speech (POS) tagging
- Chunking
- Slot filling

In [ ]:
# Let's see NER in action with spaCy
nlp = spacy.load("en_core_web_sm")

text = "Apple Inc. was founded by Steve Jobs in Cupertino, California on April 1, 1976."
doc = nlp(text)

print("📝 Original Text:")
print(f"   {text}\n")

print("🏷️ Detected Entities:")
print("-" * 60)
for ent in doc.ents:
    print(f"   {ent.text:<25} → {ent.label_:<10} ({spacy.explain(ent.label_)})")

---

## 2. Why NER Matters

### Real-World Applications

NER is a foundational NLP task that powers many applications:

| Application | How NER is Used |
|-------------|----------------|
| **Search Engines** | Understanding queries ("restaurants near Golden Gate Bridge") |
| **Customer Support** | Extracting product names, order IDs, customer names |
| **Healthcare** | Identifying drugs, diseases, symptoms from clinical notes |
| **Finance** | Extracting company names, stock tickers, monetary values |
| **Legal** | Identifying parties, dates, jurisdictions in contracts |
| **News Analysis** | Tracking mentions of people, organizations, events |
| **Knowledge Graphs** | Populating entities and relationships |
| **Chatbots/Assistants** | Understanding user intents and slot filling |

### The NER Pipeline in NLP Systems

```
┌──────────────┐    ┌─────────────┐    ┌────────────────┐    ┌──────────────┐
│  Raw Text    │ -> │ Tokenization│ -> │      NER       │ -> │ Downstream   │
│              │    │             │    │                │    │    Tasks     │
└──────────────┘    └─────────────┘    └────────────────┘    └──────────────┘
                                                                    │
                                              ┌─────────────────────┼─────────────────────┐
                                              │                     │                     │
                                              ▼                     ▼                     ▼
                                       Relation Extraction   Question Answering   Entity Linking
```

In [ ]:
# Example: Different domains have different entity needs

examples = {
    "General News": "President Biden met with Chancellor Scholz in Berlin on Tuesday.",
    "Medical": "Patient was prescribed 500mg of Metformin for Type 2 Diabetes.",
    "Finance": "Tesla (TSLA) stock rose 5% after announcing $10B revenue.",
    "Legal": "The defendant, John Smith, filed a motion in the Southern District of New York."
}

print("🌐 NER Across Different Domains\n")
for domain, text in examples.items():
    doc = nlp(text)
    print(f"📌 {domain}:")
    print(f"   Text: {text}")
    entities = [(ent.text, ent.label_) for ent in doc.ents]
    print(f"   Entities: {entities}")
    print()

---

## 3. Entity Types & Categories

### Standard Entity Types

While entity types can be customized for any domain, there are several standard categories:

#### CoNLL-2003 Entity Types (Most Common Benchmark)

| Tag | Description | Examples |
|-----|-------------|----------|
| **PER** | Person names | "Albert Einstein", "Marie Curie" |
| **ORG** | Organizations | "Google", "United Nations" |
| **LOC** | Locations | "Paris", "Mount Everest" |
| **MISC** | Miscellaneous | "Nobel Prize", "Olympics" |

> **Reference**: [CoNLL-2003 Shared Task](https://www.clips.uantwerpen.be/conll2003/ner/)

#### OntoNotes 5.0 Entity Types (18 types, used by spaCy)

| Tag | Description | Examples |
|-----|-------------|----------|
| PERSON | People, including fictional | "Barack Obama" |
| NORP | Nationalities, religious/political groups | "American", "Buddhist" |
| FAC | Buildings, airports, highways, etc. | "Empire State Building" |
| ORG | Companies, agencies, institutions | "Microsoft" |
| GPE | Countries, cities, states | "France", "Tokyo" |
| LOC | Non-GPE locations | "Pacific Ocean" |
| PRODUCT | Objects, vehicles, foods, etc. | "iPhone" |
| EVENT | Named hurricanes, battles, etc. | "World War II" |
| WORK_OF_ART | Titles of books, songs, etc. | "Hamlet" |
| LAW | Named documents made into laws | "Constitution" |
| LANGUAGE | Any named language | "French" |
| DATE | Absolute or relative dates | "January 2024" |
| TIME | Times smaller than a day | "3:00 PM" |
| PERCENT | Percentage | "25%" |
| MONEY | Monetary values | "$1 million" |
| QUANTITY | Measurements | "100 miles" |
| ORDINAL | "first", "second", etc. | "first" |
| CARDINAL | Numerals that don't fall under another type | "42" |

> **Reference**: [OntoNotes 5.0](https://catalog.ldc.upenn.edu/LDC2013T19)

In [ ]:
# Explore spaCy's entity types
print("📋 spaCy's Entity Labels (OntoNotes):\n")
print("-" * 70)

# Get all entity labels from spaCy
labels = nlp.get_pipe("ner").labels

for label in sorted(labels):
    explanation = spacy.explain(label)
    print(f"   {label:<12} → {explanation}")

In [ ]:
# Let's see all entity types in action
comprehensive_text = """
Apple CEO Tim Cook announced on January 15, 2024 that the company's 
revenue reached $100 billion, a 15% increase. The American tech giant, 
headquartered at Apple Park in Cupertino, plans to launch three new 
iPhone models. The event will be held at 10:00 AM at the Moscone Center 
in San Francisco. "This is our most innovative product line," said Cook,
who has led Apple since 2011.
"""

doc = nlp(comprehensive_text)

print("🔍 Comprehensive Entity Detection:\n")
print("-" * 70)

# Group entities by type
entities_by_type = {}
for ent in doc.ents:
    if ent.label_ not in entities_by_type:
        entities_by_type[ent.label_] = []
    entities_by_type[ent.label_].append(ent.text)

for label, entities in sorted(entities_by_type.items()):
    print(f"\n{label} ({spacy.explain(label)}):")
    for ent in entities:
        print(f"   • {ent}")

---

## 4. Tagging Schemes: BIO, BIOES, and IOB

### The Problem: Multi-Token Entities

Consider: **"New York City"** - this is a single entity (LOCATION) spanning 3 tokens.

How do we represent this? We need a way to:
1. Mark where an entity **begins**
2. Mark tokens that are **inside** an entity
3. Mark tokens that are **outside** any entity
4. Handle **consecutive entities** of the same type

### BIO (IOB2) Scheme

The most common tagging scheme:

| Prefix | Meaning |
|--------|--------|
| **B-** | **B**eginning of an entity |
| **I-** | **I**nside an entity (continuation) |
| **O** | **O**utside any entity |

#### Example:

```
Sentence: "Barack Obama was born in Honolulu"

Token:    Barack   Obama    was    born   in    Honolulu
Tag:      B-PER    I-PER    O      O      O     B-LOC
```

### Why B- is Important

Consider two consecutive person entities:

```
Sentence: "John Smith Mary Johnson met yesterday"

Without B-:
Token:    John     Smith    Mary     Johnson   met    yesterday
Tag:      PER      PER      PER      PER       O      O
          ^^^^^^^^^^^^^^^^^^^^^^^^^
          This looks like ONE entity!

With BIO:
Token:    John     Smith    Mary     Johnson   met    yesterday
Tag:      B-PER    I-PER    B-PER    I-PER     O      O
          ^^^^^^^^^^^^      ^^^^^^^^^^^^^^^^^
          Entity 1          Entity 2
```

The **B-** prefix allows us to distinguish where one entity ends and another begins!

In [ ]:
# Visualize BIO tagging
def visualize_bio_tags(tokens, tags):
    """Visualize BIO tags for a sequence"""
    print("\n" + "=" * 70)
    print("Token-Tag Alignment:")
    print("=" * 70)
    
    # Print tokens
    print("\nTokens: ", end="")
    for token in tokens:
        print(f"{token:<12}", end="")
    
    # Print tags
    print("\nTags:   ", end="")
    for tag in tags:
        print(f"{tag:<12}", end="")
    
    # Extract entities
    print("\n\n" + "-" * 70)
    print("Extracted Entities:")
    print("-" * 70)
    
    current_entity = []
    current_type = None
    entities = []
    
    for token, tag in zip(tokens, tags):
        if tag.startswith('B-'):
            # Save previous entity if exists
            if current_entity:
                entities.append((" ".join(current_entity), current_type))
            # Start new entity
            current_entity = [token]
            current_type = tag[2:]  # Remove 'B-' prefix
        elif tag.startswith('I-') and current_entity:
            # Continue current entity
            current_entity.append(token)
        else:  # O tag
            # Save previous entity if exists
            if current_entity:
                entities.append((" ".join(current_entity), current_type))
                current_entity = []
                current_type = None
    
    # Don't forget the last entity
    if current_entity:
        entities.append((" ".join(current_entity), current_type))
    
    for entity, etype in entities:
        print(f"   • '{entity}' → {etype}")

# Example 1: Simple case
tokens1 = ["Barack", "Obama", "was", "born", "in", "Honolulu"]
tags1 = ["B-PER", "I-PER", "O", "O", "O", "B-LOC"]
visualize_bio_tags(tokens1, tags1)

# Example 2: Consecutive entities
print("\n" + "#" * 70)
tokens2 = ["John", "Smith", "met", "Mary", "Johnson", "in", "New", "York"]
tags2 = ["B-PER", "I-PER", "O", "B-PER", "I-PER", "O", "B-LOC", "I-LOC"]
visualize_bio_tags(tokens2, tags2)

### Other Tagging Schemes

#### IOB1 (Original IOB)
- B- is only used when two entities of the same type are adjacent
- Otherwise, I- is used for the first token

#### BIOES (also called BILOU)

| Prefix | Meaning |
|--------|--------|
| **B-** | Beginning of multi-token entity |
| **I-** | Inside a multi-token entity |
| **O** | Outside any entity |
| **E-** | End of multi-token entity |
| **S-** | Single-token entity |

```
BIO:    B-PER   I-PER   I-PER   O    B-LOC
BIOES:  B-PER   I-PER   E-PER   O    S-LOC
        ^
        John    Paul    Jones   met  Paris
```

> **Research Note**: Some studies show BIOES can improve performance by 0.5-1% F1 score by providing more explicit boundary information.
>
> Reference: [Ratinov & Roth, 2009](https://aclanthology.org/W09-1119/)

In [ ]:
# Compare BIO vs BIOES
def compare_tagging_schemes(tokens):
    bio_tags = ["B-PER", "I-PER", "I-PER", "O", "B-LOC", "O", "B-ORG", "I-ORG"]
    bioes_tags = ["B-PER", "I-PER", "E-PER", "O", "S-LOC", "O", "B-ORG", "E-ORG"]
    
    print("Comparison of BIO vs BIOES Tagging Schemes")
    print("=" * 70)
    print(f"{'Token':<12} {'BIO':<12} {'BIOES':<12}")
    print("-" * 70)
    for token, bio, bioes in zip(tokens, bio_tags, bioes_tags):
        print(f"{token:<12} {bio:<12} {bioes:<12}")

tokens = ["John", "Paul", "Jones", "visited", "Paris", "with", "Apple", "Inc"]
compare_tagging_schemes(tokens)

---

## 5. Evolution of NER Approaches

### Historical Timeline

```
1990s          2000s           2010s           2018+           2023+
  │              │               │               │               │
  ▼              ▼               ▼               ▼               ▼
Rule-based → Statistical   → Neural/LSTM → Transformer  → Zero-shot
             (CRF, HMM)      (BiLSTM-CRF)    (BERT)        (GLiNER)
```

### 1. Rule-Based Systems (1990s)

**Approach**: Hand-crafted rules and gazetteers (lists of known entities)

```python
# Example pseudo-rule
if token.istitle() and previous_token in ['Mr.', 'Mrs.', 'Dr.']:
    label = 'PERSON'
```

**Pros**: Precise, interpretable, no training data needed
**Cons**: Doesn't generalize, expensive to maintain

### 2. Statistical Models (2000s)

**Key Models**:
- Hidden Markov Models (HMM)
- Maximum Entropy Markov Models (MEMM)
- **Conditional Random Fields (CRF)** ← Most successful

**Features used**:
- Word itself
- Capitalization patterns
- Part-of-speech tags
- Gazetteer membership
- Word shapes (e.g., "Xxxx" for "John")

> **Reference**: [Lafferty et al., 2001 - CRFs for Sequence Labeling](https://repository.upenn.edu/cgi/viewcontent.cgi?article=1162&context=cis_papers)

### 3. Neural Networks (2015-2018)

**Architecture**: BiLSTM + CRF

```
Input: [w1, w2, w3, ...]
         ↓
    Word Embeddings (Word2Vec, GloVe)
         ↓
    Character-level CNN/LSTM
         ↓
    Bidirectional LSTM
         ↓
    CRF Layer (for valid tag sequences)
         ↓
Output: [B-PER, I-PER, O, ...]
```

> **Reference**: [Lample et al., 2016 - Neural NER](https://arxiv.org/abs/1603.01360)

### 4. Transformer Era (2018+)

**BERT for NER**: Fine-tune pretrained language model with a token classification head.

```
Input: [CLS] Barack Obama was born in Honolulu [SEP]
         ↓
    BERT Encoder (12-24 layers)
         ↓
    Token representations
         ↓
    Linear Classification Layer
         ↓
Output: [_, B-PER, I-PER, O, O, O, B-LOC, _]
```

> **Reference**: [Devlin et al., 2019 - BERT](https://arxiv.org/abs/1810.04805)

### 5. Zero-Shot NER (2023+)

**GLiNER**: Extract ANY entity type without training!

```
Input: 
  - Text: "Apple released the iPhone in 2007"
  - Labels: ["company", "product", "year"]
         ↓
    GLiNER Model
         ↓
Output: Apple→company, iPhone→product, 2007→year
```

> **Reference**: [Zaratiana et al., 2023 - GLiNER](https://arxiv.org/abs/2311.08526)

In [ ]:
# Performance comparison across eras (approximate CoNLL-2003 F1 scores)
import matplotlib.pyplot as plt

models = [
    ("Rule-based", 1996, 70),
    ("CRF", 2003, 85),
    ("BiLSTM-CRF", 2016, 91),
    ("BERT-base", 2019, 92.4),
    ("BERT-large", 2019, 93.5),
    ("RoBERTa", 2020, 93.8),
    ("DeBERTa", 2021, 94.6),
]

names = [m[0] for m in models]
years = [m[1] for m in models]
scores = [m[2] for m in models]

plt.figure(figsize=(12, 6))
colors = plt.cm.viridis([i/len(models) for i in range(len(models))])
bars = plt.bar(names, scores, color=colors)

plt.ylabel('F1 Score (%)', fontsize=12)
plt.title('Evolution of NER Performance on CoNLL-2003', fontsize=14)
plt.ylim(60, 100)
plt.xticks(rotation=45, ha='right')

# Add value labels on bars
for bar, score in zip(bars, scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{score}%', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

print("\n📈 Key Observations:")
print("   • CRF brought a ~15% improvement over rule-based systems")
print("   • Neural models (BiLSTM) added another ~6%")
print("   • BERT and transformers pushed beyond 93%")
print("   • We're now approaching human-level performance (~97%)")

---

## 6. Hands-on: Basic NER Exploration

Let's explore the CoNLL-2003 dataset, the most widely used NER benchmark.

In [ ]:
# Load CoNLL-2003 dataset
dataset = load_dataset("eriktks/conll2003", trust_remote_code=True)

print("📊 CoNLL-2003 Dataset Overview")
print("=" * 50)
print(f"\nDataset splits:")
for split in dataset:
    print(f"   • {split}: {len(dataset[split])} examples")

print(f"\nFeatures: {list(dataset['train'].features.keys())}")

In [ ]:
# Examine the tag set
ner_tags = dataset['train'].features['ner_tags'].feature
print("🏷️ NER Tag Set:")
print("-" * 40)
for i, name in enumerate(ner_tags.names):
    print(f"   {i}: {name}")

In [ ]:
# Look at a sample
sample = dataset['train'][0]

print("📝 Sample from CoNLL-2003:\n")
print("Tokens:", sample['tokens'])
print("\nNER Tags (as indices):", sample['ner_tags'])

# Convert to tag names
tag_names = [ner_tags.names[tag] for tag in sample['ner_tags']]
print("\nNER Tags (as names):", tag_names)

print("\n" + "-" * 60)
print("\nAligned view:")
print(f"{'Token':<20} {'Tag':<10}")
print("-" * 30)
for token, tag in zip(sample['tokens'], tag_names):
    print(f"{token:<20} {tag:<10}")

In [ ]:
# Analyze entity distribution in the dataset
from collections import Counter

# Count entities in training set
entity_counts = Counter()
tag_names_list = ner_tags.names

for example in dataset['train']:
    for tag_id in example['ner_tags']:
        tag = tag_names_list[tag_id]
        if tag != 'O':
            # Extract entity type (remove B- or I- prefix)
            entity_type = tag.split('-')[1] if '-' in tag else tag
            entity_counts[entity_type] += 1

print("📊 Entity Distribution in CoNLL-2003 Training Set:\n")
total = sum(entity_counts.values())
for entity_type, count in entity_counts.most_common():
    pct = count / total * 100
    bar = '█' * int(pct / 2)
    print(f"   {entity_type:<6} {count:>6} ({pct:>5.1f}%) {bar}")

In [ ]:
# Find and display examples with multiple entity types
print("🔍 Examples with Multiple Entity Types:\n")

count = 0
for example in dataset['train']:
    tags = [tag_names_list[t] for t in example['ner_tags']]
    unique_entity_types = set([t.split('-')[1] for t in tags if t != 'O'])
    
    if len(unique_entity_types) >= 3:  # At least 3 different entity types
        print(f"Sentence: {' '.join(example['tokens'])}")
        print(f"Entity types found: {unique_entity_types}")
        
        # Show entities
        current_entity = []
        current_type = None
        
        for token, tag in zip(example['tokens'], tags):
            if tag.startswith('B-'):
                if current_entity:
                    print(f"   • {' '.join(current_entity)} → {current_type}")
                current_entity = [token]
                current_type = tag[2:]
            elif tag.startswith('I-'):
                current_entity.append(token)
            else:
                if current_entity:
                    print(f"   • {' '.join(current_entity)} → {current_type}")
                    current_entity = []
                    current_type = None
        
        if current_entity:
            print(f"   • {' '.join(current_entity)} → {current_type}")
        
        print()
        count += 1
        if count >= 3:
            break

---

## 7. Further Reading & Resources

### 📚 Essential Papers

1. **CRF for Sequence Labeling** (2001)
   - Lafferty, J., McCallum, A., & Pereira, F.
   - [Paper](https://repository.upenn.edu/cgi/viewcontent.cgi?article=1162&context=cis_papers)

2. **Neural Architectures for NER** (2016)
   - Lample, G., Ballesteros, M., Subramanian, S., Kawakami, K., & Dyer, C.
   - [arXiv:1603.01360](https://arxiv.org/abs/1603.01360)

3. **BERT: Pre-training of Deep Bidirectional Transformers** (2019)
   - Devlin, J., Chang, M. W., Lee, K., & Toutanova, K.
   - [arXiv:1810.04805](https://arxiv.org/abs/1810.04805)

4. **GLiNER: Generalist Model for Named Entity Recognition** (2023)
   - Zaratiana, U., Tomeh, N., Holat, P., & Charnois, T.
   - [arXiv:2311.08526](https://arxiv.org/abs/2311.08526)

### 🔗 Official Documentation

- [spaCy NER Documentation](https://spacy.io/usage/linguistic-features#named-entities)
- [Hugging Face Token Classification](https://huggingface.co/docs/transformers/tasks/token_classification)
- [CoNLL-2003 Shared Task](https://www.clips.uantwerpen.be/conll2003/ner/)

### 📊 Benchmarks & Datasets

- [CoNLL-2003 on Hugging Face](https://huggingface.co/datasets/eriktks/conll2003)
- [OntoNotes 5.0](https://catalog.ldc.upenn.edu/LDC2013T19)
- [WNUT-17 (Emerging Entities)](https://huggingface.co/datasets/wnut_17)

### 🎓 Tutorials & Courses

- [Hugging Face NLP Course - Chapter 7](https://huggingface.co/learn/llm-course/en/chapter7/2)
- [Stanford CS224N - NER Lecture](http://web.stanford.edu/class/cs224n/)

---

## ✅ Lesson Summary

In this lesson, we covered:

1. **What NER is**: Identifying and classifying named entities in text
2. **Why it matters**: Foundation for many NLP applications
3. **Entity types**: From CoNLL-2003's 4 types to OntoNotes' 18 types
4. **Tagging schemes**: BIO, IOB, BIOES and why they're needed
5. **Evolution**: From rules → CRF → Neural → Transformers → Zero-shot

### 🚀 Next Lesson Preview

In **Lesson 2**, we'll dive deep into **spaCy's NER system**, including:
- Using pre-trained models
- Custom entity rulers
- Visualizing entities
- Training custom NER models

In [ ]:
print("🎉 Congratulations! You've completed Lesson 1: NER Fundamentals")
print("\n📝 Key takeaways:")
print("   1. NER is a sequence labeling task")
print("   2. BIO tagging handles multi-token entities")
print("   3. Modern transformers achieve >93% F1 on CoNLL-2003")
print("   4. Zero-shot models like GLiNER don't need training data")
print("\n👉 Continue to Lesson 2: Traditional NER with spaCy")